In [1]:
!pip install  pyspark

In [2]:
import os
import shutil

import pandas as pd
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, IntegerType, StringType, StructField, StructType

In [3]:
spark = (SparkSession.builder.appName("WarehouseInsights").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

In [5]:
uploaded = files.upload()
print(list(uploaded.keys()))

Saving stock_movements_clean.csv to stock_movements_clean (1).csv
['stock_movements_clean (1).csv']


In [6]:
uploaded = files.upload()
print(list(uploaded.keys()))

Saving products.csv to products.csv
Saving stock_movements_dirty.csv to stock_movements_dirty.csv
Saving suppliers.csv to suppliers.csv
Saving warehouses.csv to warehouses.csv
['products.csv', 'stock_movements_dirty.csv', 'suppliers.csv', 'warehouses.csv']


In [7]:
movement_schema = StructType([
    StructField("movement_id",   IntegerType(), False),
    StructField("product_id",    IntegerType(), False),
    StructField("warehouse_id",  IntegerType(), False),
    StructField("movement_type", StringType(),  True),
    StructField("quantity",      IntegerType(), False),
    StructField("movement_date", DateType(),    True),
    StructField("reference_no",  StringType(),  True),
    StructField("reason",        StringType(),  True),
])

movements  = spark.read.csv("stock_movements_clean.csv", header=True, schema=movement_schema)
products   = spark.read.csv("products.csv",   header=True, inferSchema=True)
warehouses = spark.read.csv("warehouses.csv", header=True, inferSchema=True)

print("movements:", movements.count(), "| products:", products.count(), "| warehouses:", warehouses.count())
movements.show(5)

movements: 200 | products: 10 | warehouses: 3
+-----------+----------+------------+-------------+--------+-------------+------------+---------------+
|movement_id|product_id|warehouse_id|movement_type|quantity|movement_date|reference_no|         reason|
+-----------+----------+------------+-------------+--------+-------------+------------+---------------+
|       1004|         2|           1|           IN|     123|   2026-09-01|     PO-1004|Opening receipt|
|       1005|         2|           2|           IN|      22|   2026-09-01|     PO-1005|Opening receipt|
|       1006|         2|           3|           IN|      26|   2026-09-01|     PO-1006|Opening receipt|
|       1012|         4|           3|           IN|      67|   2026-09-01|     PO-1012|Opening receipt|
|       1014|         5|           2|           IN|      27|   2026-09-01|     PO-1014|Opening receipt|
+-----------+----------+------------+-------------+--------+-------------+------------+---------------+
only showing top 5

In [8]:
stock = movements.groupBy("product_id", "warehouse_id").agg(
    F.sum("quantity").alias("quantity_on_hand"),
    F.max("movement_date").alias("last_movement_date"))

detail = (stock
          .join(F.broadcast(products), "product_id")
          .join(F.broadcast(warehouses.select("warehouse_id", "warehouse_name", "city")), "warehouse_id")
          .withColumn("stock_status",
                      F.when(F.col("quantity_on_hand") < F.col("reorder_level"), "LOW_STOCK")
                       .when(F.col("quantity_on_hand") > F.col("max_stock_level"), "OVERSTOCK")
                       .otherwise("OK"))
          .withColumn("stock_value", F.col("quantity_on_hand") * F.col("unit_price")))
detail.cache()

print("Product-warehouse rows:", detail.count())
detail.select("warehouse_name", "sku", "product_name", "quantity_on_hand",
              "reorder_level", "max_stock_level", "stock_status").show()

Product-warehouse rows: 30
+------------------+--------+--------------------+----------------+-------------+---------------+------------+
|    warehouse_name|     sku|        product_name|quantity_on_hand|reorder_level|max_stock_level|stock_status|
+------------------+--------+--------------------+----------------+-------------+---------------+------------+
|Chennai Central WH|ELE-1002|   USB-C Charger 65W|             185|           20|            200|          OK|
|Bengaluru North WH|ELE-1002|   USB-C Charger 65W|             113|           20|            200|          OK|
|Bengaluru North WH|GRO-5001|    Basmati Rice 5kg|             485|           40|            400|   OVERSTOCK|
| Hyderabad East WH|ELE-1001|      Wireless Mouse|             110|           25|            300|          OK|
| Hyderabad East WH|ELE-1002|   USB-C Charger 65W|              12|           20|            200|   LOW_STOCK|
|Chennai Central WH|STA-3002|   Gel Pen Pack (10)|              80|           60|    

In [9]:
total_per_warehouse = (movements.groupBy("warehouse_id")
                       .agg(F.sum("quantity").alias("total_units"))
                       .join(warehouses.select("warehouse_id", "warehouse_name"), "warehouse_id")
                       .orderBy("warehouse_id"))
total_per_warehouse.show()

+------------+-----------+------------------+
|warehouse_id|total_units|    warehouse_name|
+------------+-----------+------------------+
|           1|       1589|Chennai Central WH|
|           2|       2030|Bengaluru North WH|
|           3|       1091| Hyderabad East WH|
+------------+-----------+------------------+



In [10]:
THRESHOLD_PCT = 25.0

warehouse_status = (detail.groupBy("warehouse_id", "warehouse_name", "city").agg(
        F.sum("quantity_on_hand").alias("total_units"),
        F.round(F.sum("stock_value"), 2).alias("total_stock_value"),
        F.count("*").alias("product_lines"),
        F.sum(F.when(F.col("stock_status") == "LOW_STOCK", 1).otherwise(0)).alias("low_stock_items"),
        F.sum(F.when(F.col("stock_status") == "OVERSTOCK", 1).otherwise(0)).alias("overstock_items"))
    .withColumn("low_stock_pct",  F.round(100 * F.col("low_stock_items")  / F.col("product_lines"), 1))
    .withColumn("overstock_pct",  F.round(100 * F.col("overstock_items") / F.col("product_lines"), 1))
    .withColumn("warehouse_status",
                F.when((F.col("low_stock_pct") >= THRESHOLD_PCT) &
                       (F.col("low_stock_pct") >= F.col("overstock_pct")), "UNDERSTOCKED")
                 .when(F.col("overstock_pct") >= THRESHOLD_PCT, "OVERSTOCKED")
                 .otherwise("BALANCED"))
    .orderBy("warehouse_id"))

warehouse_status.show(truncate=False)

+------------+------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|warehouse_id|warehouse_name    |city     |total_units|total_stock_value|product_lines|low_stock_items|overstock_items|low_stock_pct|overstock_pct|warehouse_status|
+------------+------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|1           |Chennai Central WH|Chennai  |1589       |3130436.0        |10           |0              |4              |0.0          |40.0         |OVERSTOCKED     |
|2           |Bengaluru North WH|Bengaluru|2030       |2114545.0        |10           |0              |2              |0.0          |20.0         |BALANCED        |
|3           |Hyderabad East WH |Hyderabad|1091       |2342051.0        |10           |2              |2              |20.0         |20.0         |BALANCED        |
+---------

In [11]:
exceptions = (detail.filter(F.col("stock_status") != "OK")
              .select("warehouse_name", "sku", "product_name", "category",
                      "quantity_on_hand", "reorder_level", "max_stock_level", "stock_status")
              .orderBy("warehouse_name", "stock_status", "sku"))

print("Items needing attention:", exceptions.count())
print("\nUNDERSTOCKED items (below reorder level):")
exceptions.filter(F.col("stock_status") == "LOW_STOCK").show(10)
print("OVERSTOCKED items (above maximum level):")
exceptions.filter(F.col("stock_status") == "OVERSTOCK").show(10)

Items needing attention: 10

UNDERSTOCKED items (below reorder level):
+-----------------+--------+-------------------+-----------+----------------+-------------+---------------+------------+
|   warehouse_name|     sku|       product_name|   category|quantity_on_hand|reorder_level|max_stock_level|stock_status|
+-----------------+--------+-------------------+-----------+----------------+-------------+---------------+------------+
|Hyderabad East WH|ELE-1002|  USB-C Charger 65W|Electronics|              12|           20|            200|   LOW_STOCK|
|Hyderabad East WH|STA-3001|A4 Paper Ream (500)| Stationery|              38|           50|            500|   LOW_STOCK|
+-----------------+--------+-------------------+-----------+----------------+-------------+---------------+------------+

OVERSTOCKED items (above maximum level):
+------------------+--------+--------------------+-----------+----------------+-------------+---------------+------------+
|    warehouse_name|     sku|        p

In [12]:
os.makedirs("output", exist_ok=True)
warehouse_status.toPandas().to_csv("output/warehouse_stock_status.csv", index=False)
exceptions.toPandas().to_csv("output/warehouse_item_exceptions.csv", index=False)

pd.read_csv("output/warehouse_stock_status.csv")

,warehouse_id,warehouse_name,city,total_units,total_stock_value,product_lines,low_stock_items,overstock_items,low_stock_pct,overstock_pct,warehouse_status
0,1,Chennai Central WH,Chennai,1589,3130436.0,10,0,4,0.0,40.0,OVERSTOCKED
1,2,Bengaluru North WH,Bengaluru,2030,2114545.0,10,0,2,0.0,20.0,BALANCED
2,3,Hyderabad East WH,Hyderabad,1091,2342051.0,10,2,2,20.0,20.0,BALANCED


In [13]:
shutil.make_archive("week3_output", "zip", "output")
files.download("week3_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>